In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
import networkx as nx
from typing import Dict, List

def build_article_graph(claim_text: str) -> nx.Graph:
    """
    Constructs the base intra-article graph for AVeriTeC.
    Isolated claims become a single root node.
    """
    G = nx.Graph()
    G.add_node(
        0,
        text=claim_text,
        role='claim',
        is_evidence=0,
        source_domain='target'
    )
    return G

def add_evidence_nodes(base_graph: nx.Graph, retrieved_docs: List[Dict], cred_db: Dict) -> nx.Graph:
    """Appends external web documents to the graph before NLI edge generation."""
    G = base_graph.copy()
    current_node_id = max(G.nodes) + 1

    for doc in retrieved_docs:
        domain = doc.get('source_domain', 'unknown')
        weight = cred_db.get(domain, 0.5)

        G.add_node(
            current_node_id,
            text=doc['text'],
            role='evidence',
            is_evidence=1,
            source_domain=domain,
            node_weight=weight
        )
        current_node_id += 1

    return G

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train = vectorizer.fit_transform(train_df['text'])
y_train = train_df['label']
X_test = vectorizer.transform(test_df['text'])
y_test = test_df['label']

baseline_clf = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced')
baseline_clf.fit(X_train, y_train)

preds = baseline_clf.predict(X_test)
probs = baseline_clf.predict_proba(X_test)[:, 1]
print("=== BASELINE RESULTS ===")
print(classification_report(y_test, preds))
print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f}")

NameError: name 'train_df' is not defined

In [ ]:
import torch
from torch_geometric.loader import DataLoader
from pipeline.retrieval import search_claim_evidence
from pipeline.graph_building import build_article_graph, add_evidence_nodes
from pipeline.augmentation import augment_with_cross_source
from pipeline.features import graph_to_pyg
from pipeline.model import FakeNewsGAT

# Mock DB mapping domains to Beta prior credibility scores
credibility_db = {}

def build_claim_graph(claim_text, speaker):
    base_graph = build_article_graph(claim_text)
    evidence_docs = search_claim_evidence(claim_text, speaker=speaker, max_results=3)
    populated_graph = add_evidence_nodes(base_graph, evidence_docs, credibility_db)

    augmented_graph = augment_with_cross_source(
        base_graph=populated_graph,
        headline=claim_text,
        cred_db=credibility_db
    )
    return graph_to_pyg(augmented_graph)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gat_model = FakeNewsGAT(in_channels=20, hidden_channels=64, out_channels=1).to(device)

print("\nBuilding training graphs (executing DuckDuckGo searches)...")
train_graphs = []
for _, row in train_df.head(50).iterrows(): # Truncated for immediate testing
    try:
        g = build_claim_graph(row['text'], row['speaker'])
        g.y = torch.tensor([row['label']], dtype=torch.float)
        train_graphs.append(g)
    except Exception:
        continue

train_loader = DataLoader(train_graphs, batch_size=16, shuffle=True)
optimizer = torch.optim.Adam(gat_model.parameters(), lr=0.001)
criterion = torch.nn.BCEWithLogitsLoss()

gat_model.train()
for epoch in range(10):
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = gat_model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out.squeeze(), batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/10 | Loss: {total_loss/len(train_loader):.4f}")